# JSON report reference

The field-level reference for the JSON report, schema version `0.2`.
The schema version identifies the report format. For a fixed package
version, the same data and configuration produce byte-identical JSON
when neither a deadline nor timing is enabled. Code that reads reports should check the
schema version and tolerate unknown keys.

The top level:


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Field': ['`schema_version`', '`overview`', '`columns`', '`findings`', '`groups`', '`selection`', '`coverage`'], 'Type': ['string', 'object', 'object', 'list', 'list', 'object', 'object'], 'Contents': ['`"0.2"`', '`n_rows`, `n_columns`, `headline`, `dtype_counts`, `role_counts`, `top_findings` (top 5 claims)', 'one card per column, keyed by name (columns)', 'selected findings in rank order', 'connected findings over shared columns; no redundancy claim', '`candidates`, `verified_candidates`, `distinct_statements`, `selected`', 'per-pass `status`, `detail`, and optional `elapsed_ms` (coverage)']})
(GT(_doc_table)
 .fmt_markdown(columns=['Field', 'Type', 'Contents'])
 .tab_header(title='JSON report reference')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


See [columns](../usage/columns.ipynb), [coverage](../usage/coverage.ipynb) for the related field descriptions.


## Finding fields


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Field': ['`id`', '`category`', '`type`', '`subject`', '`specification`', '`claim`', '`evidence`', '`interestingness`', '`interpretation`'], 'Type': ['string', 'string', 'string', 'object', 'object', 'string', 'list', 'float', 'string?'], 'Meaning': ['`category-<fnv1a64>` of the canonical statement — stable across reruns, independent of rank and detector', 'one of `missingness`, `concentration`, `column_roles`, `uniqueness`, `duplicates`, `formats`, `dependencies`, `conditional_rules`, `associations`, `gaps`, `intervals`, `ordering`, `changes`', 'Always present: the subtype within a category; examples include `functional`, `predictive`, `if_then`, `period_association`, and `distribution_shift`', '`columns` (all involved); dependencies add `inputs` and `target`; comparisons add `unit`', '`form` (`value_mapping`, `affine`, `piecewise_linear`), `conditions` and `outcome` for rules, `definition` for comparison settings such as tolerances and calendar references', 'text summary of the result', 'one record per contributing method — see below', 'deterministic selection priority in $[0,1]$; not a probability', 'a suggested implication, when one exists']})
(GT(_doc_table)
 .fmt_markdown(columns=['Field', 'Type', 'Meaning'])
 .tab_header(title='Finding fields')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


Predicates (rule conditions and outcomes) carry `column`, `operator`
(`eq`, `ne`, `lt`, `le`, `gt`, `ge`), a typed scalar `value`, and an
optional `transform` (`day_index` for date comparisons). Numeric values
stay numbers; conjunctions are lists.


## Evidence records


In [ ]:
import polars as pl
from great_tables import GT, loc, style

_doc_table = pl.DataFrame({'Field': ['`method`', '`basis`', '`scope`', '`validation`', '`metrics`'], 'Type': ['object', 'string', 'object', 'object', 'object'], 'Meaning': ['`detector` (registry id), `algorithm`, `provenance` (implementation source)', '`measured`, `estimated`, or `inferred`', '`sampling` (`full` / `sample` / `unspecified`), `population_rows`, `rows_evaluated`, `missing_values` policy', '`method` (`direct_comparison`, `cross_validation`, `diagnostic_holdout`, `diagnostic_partition`, `none`), `training_rows`, `evaluation_rows`, `folds`, `used_for_selection`', 'everything the method measured — counts, rates, coefficients, thresholds; detector-specific keys stay detector-specific']})
(GT(_doc_table)
 .fmt_markdown(columns=['Field', 'Type', 'Meaning'])
 .tab_header(title='Evidence records')
 .tab_options(table_width="100%", table_layout="fixed",
              table_font_size="14px", data_row_padding="9px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


When methods produce the same normalized statement, the report combines
their evidence in one finding. It retains each method's counts and rates
without averaging them. For example, a rule miner may count matches in its
training rows while the shared evaluator counts matches in the diagnostic
partition. Use each record's `scope` and `validation` to identify the rows
behind its metrics.


## A complete finding

The example below generates a finding from the installed package:

In [ ]:
import json

import polars as pl

import onceover

df = pl.DataFrame({
    "quantity": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10] * 30,
})
df = df.with_columns((2 * pl.col("quantity")).alias("double_quantity"))

report = onceover.profile(df)
identity = next(f for f in report.findings
                if f["specification"].get("form") == "affine")
print(json.dumps(identity, indent=2))

## Selection defaults

The default selection policy retains at most 100 findings across all
detectors, including at most 20 conditional rules, 2 rules per outcome,
and 3 predictive dependencies per target. To qualify for selection,
a rule must meet the shared evaluator's thresholds: ≥ 30 eligible rows,
≥ 30 antecedent matches, ≥ 20 joint matches, conditional rate above baseline by ≥ 0.02 absolute or lift ≥
1.5. Python exposes `max_findings` and `max_rules`; the
Rust `SelectionPolicy` configures the rest.

## Versioning

`schema_version` identifies the format expected by a reader. Version 0.2
replaced detector-named finding kinds with semantic statements during
pre-release development. Readers should check the version before
interpreting fields and tolerate unknown keys within a supported format.